# Synthetic-data model training

This notebook evaluates MLP, GNN, and XGBoost models on synthetic and HIGGS datasets.
It configures the data generator, checks interaction-graph recovery, and runs a tracked
hyperparameter sweep with a fixed validation and test protocol.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
import warnings

import dotenv
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
from lightning.pytorch import seed_everything

from my_project.data.generators import (
    DEFAULT_INTERACTIONS,
    build_generator,
    compute_config_hash,
    interactions_name_map,
)
from my_project.data.utils import prepare_data_fixed_test
from my_project.experiments.synthetic import (
    GNN_SEARCH_SPACE,
    MLP_SEARCH_SPACE,
    XGB_SEARCH_SPACE,
    higgs_pipeline,
    higgs_pipeline_discretized,
)
from my_project.gnn.models import MyGNN, MyMLP
from my_project.graphs import build_ii_graphs, build_true_graph, graph_stats
from my_project.information_theory import compute_info
from my_project.training.sweep import build_generator_name, run_sweep

In [ ]:
EXPERIMENT_SETTINGS = {
    "seed": 12345,
    "data_seed": 1234,
    "data_type": "synthetic",
    "n_startup_trials": 10,
    "n_trials": 50,
}

N_STARTUP_TRIALS = EXPERIMENT_SETTINGS["n_startup_trials"]
N_TRIALS = EXPERIMENT_SETTINGS["n_trials"]

N_BINS = None

SEED = EXPERIMENT_SETTINGS["seed"]
DATA_SEED = EXPERIMENT_SETTINGS["data_seed"]
DATA_TYPE = EXPERIMENT_SETTINGS["data_type"]
SAMPLER_SEED = DATA_SEED

# Load environment variables explicitly so the notebook also works with nbconvert,
# plain Jupyter, and editors that do not automatically load .env files.
dotenv.load_dotenv(override=True)

# Persist Optuna studies so interrupted sweeps can resume without recomputing
# completed or pruned trials. Set this to None to use a fresh in-memory study.
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE_URL") or (
    f"sqlite:///{os.path.abspath('optuna_studies.db')}"
)

In [ ]:
seed_everything(SEED, workers=True)

# The grouped TPE sampler is intentionally experimental.
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
warnings.filterwarnings(
    "ignore",
    message=r".*isinstance\(treespec, LeafSpec\).*is deprecated.*",
)

# These in-memory datasets deliberately use a single-process data loader.
warnings.filterwarnings(
    "ignore",
    message=r".*does not have many workers.*",
)

# Suppress repetitive Trainer tips and device-status messages during the sweep.
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

USE_CUDA = torch.cuda.is_available()
KEEP_ON_GPU = True
STANDARDIZE = True

## Model and training configuration

Each model defines its own search space. The shared settings below control training,
early stopping, MLflow logging, precision, and device placement for the full sweep.

In [ ]:
CONV_LAYER = "GATConv"

# Each model has an independent search space and convolution configuration.
MODEL_CONFIGS = [
    {
        "model_cls": MyMLP,
        "convs": [None],
        "search_space": MLP_SEARCH_SPACE,
    },
    # {
    #     "model_cls": MyGNNConcat,
    #     "convs": [CONV_LAYER],
    #     "search_space": GNN_SEARCH_SPACE,
    # },
    {
        "model_cls": MyGNN,
        "convs": [CONV_LAYER],
        "search_space": GNN_SEARCH_SPACE,
    },
]

MAX_EPOCHS = 200
MONITOR_METRIC = "val/auc"
MONITOR_MODE = "max"
PRECISION = "16-mixed" if USE_CUDA else "32-true"
DIRECTION = "minimize" if MONITOR_MODE == "min" else "maximize"

# Synthetic and medical experiments keep separate search spaces so their tuning
# configurations can evolve independently.
technical_settings = {
    "trainer_kwargs": {
        "enable_progress_bar": False,
        "enable_model_summary": False,
        "precision": PRECISION,
        "max_epochs": MAX_EPOCHS,
        "accelerator": "gpu" if USE_CUDA else "cpu",
    },
    "monitor_kwargs": {"monitor": MONITOR_METRIC, "mode": MONITOR_MODE},
    "early_stopping_kwargs": {
        "patience": 10,
        "monitor": MONITOR_METRIC,
        "mode": MONITOR_MODE,
    },
    "logger_kwargs": {"tracking_uri": os.getenv("MLFLOW_TRACKING_URI")},
    "keep_on_gpu": KEEP_ON_GPU,
}

## Data generator configuration

Select one preset with `ACTIVE_GENERATOR_KEY`. The preset controls data generation,
sample-size grids, graph construction, preprocessing, and train/validation/test sizes.

In [ ]:
ACTIVE_GENERATOR_KEY = "pairwise"

EXPERIMENT_NAME = ACTIVE_GENERATOR_KEY

In [ ]:
# Selecting ACTIVE_GENERATOR_KEY configures all generator arguments, grids,
# graph settings, and random seeds below.
TOP_K = 1
DATA_SEED_MAIN = 42

# Use a separate seed stream for the shared test set.
TEST_SEED = 99991

pairwise_interactions = dict(DEFAULT_INTERACTIONS)

GENERATOR_PRESETS = {
    "pairwise": {
        "generator_kind": "pairwise",
        "f_function_id": 0,
        "n_bins": 5,
        "generator_kwargs": {
            "n_informative": 10,
            "coef_scale": 0.0,
            "coef_loc": 1.0,
            "interactions": pairwise_interactions,
        },
        "interactions_names": interactions_name_map(pairwise_interactions),
        "data_seeds": list(range(10)),
        "permute_seeds": list(range(10)),
        "train_size_grid": [
            1000,
            5000,
            10000,
        ],
        "cov_grid": [
            0.0,
            0.25,
            0.5,
        ],
        "noise_grid": None,
        "edge_mode": "n_edges",
        "n_edges_grid": [
            # 6,
            12
        ],
        "threshold_grid": [],
        "feature_selection": None,
        "test_size": 10000,
        "valid_size": 1000,
        "pipeline": None,
        "pipeline_discretized": None,
    },
    "higgs": {
        "generator_kind": "higgs",
        "f_function_id": 0,
        "n_bins": 5,
        "generator_kwargs": {},
        "interactions_names": None,
        "data_seeds": list(range(10)),
        "permute_seeds": list(range(10)),
        "train_size_grid": [10000, 5000, 1000],
        "cov_grid": [0],
        "noise_grid": None,
        "edge_mode": "threshold",
        "n_edges_grid": [],
        "threshold_grid": [0.7, 0.8, 0.9],
        "feature_selection": None,
        "test_size": 10000,
        "valid_size": 1000,
        "pipeline": higgs_pipeline,
        "pipeline_discretized": higgs_pipeline_discretized,
    },
}

In [ ]:
active_cfg = GENERATOR_PRESETS[ACTIVE_GENERATOR_KEY]

GENERATOR_KIND = active_cfg["generator_kind"]
F_FUNCTION_ID = active_cfg["f_function_id"]
N_BINS = active_cfg["n_bins"]
GENERATOR_KWARGS = active_cfg["generator_kwargs"]
INTERACTIONS_NAMES = active_cfg["interactions_names"]
DATA_SEEDS = active_cfg["data_seeds"]
PERMUTE_SEEDS = active_cfg["permute_seeds"]
TRAIN_SIZE_GRID = active_cfg["train_size_grid"]
COV_GRID = active_cfg["cov_grid"]
NOISE_GRID = active_cfg["noise_grid"]
EFFECTIVE_NOISE_GRID = [0.0] if NOISE_GRID is None else NOISE_GRID
EDGE_MODE = active_cfg["edge_mode"]
N_EDGES_GRID = active_cfg["n_edges_grid"]
THRESHOLD_GRID = active_cfg["threshold_grid"]
FEATURE_SELECTION = active_cfg["feature_selection"]
TEST_SIZE = active_cfg["test_size"]
VALID_SIZE = active_cfg["valid_size"]
PIPELINE = active_cfg.get("pipeline", None)
PIPELINE_DISCRETIZED = active_cfg.get("pipeline_discretized", None)

### Instantiate the selected generator

The cells below unpack the active preset, construct the generator, and derive stable
names and hashes used by Optuna and MLflow.

In [ ]:
generate = build_generator(GENERATOR_KIND, GENERATOR_KWARGS)

In [ ]:
GENERATOR_NAME = build_generator_name(
    GENERATOR_KIND,
    generator_kwargs=GENERATOR_KWARGS,
    f_function_id=F_FUNCTION_ID,
)

# Give every distinct generator configuration its own Optuna study.
hash_noise_level = EFFECTIVE_NOISE_GRID[0] if EFFECTIVE_NOISE_GRID else 0.0
GENERATOR_HASH = compute_config_hash(
    GENERATOR_KIND,
    GENERATOR_KWARGS,
    hash_noise_level,
    N_BINS,
)
print(f"GENERATOR_HASH = {GENERATOR_HASH}")

In [ ]:
def data_gen_params(n_samples, data_seed, noise_level=None, **gen_kwargs):
    """Return effective, MLflow-compatible parameters for one generated dataset."""
    params = {
        **GENERATOR_KWARGS,
        "n_samples": n_samples,
        "random_state": data_seed,
        "noise_level": noise_level or 0.0,
        **gen_kwargs,
    }
    if "interactions" in params:
        params["interactions"] = {
            pair: getattr(func, "__name__", str(func))
            for pair, func in params["interactions"].items()
        }
    return params

## Interaction-graph recovery diagnostics

Evaluate how well interaction-information graphs recover the known synthetic edges
across training sizes, covariance levels, and noise levels.

In [ ]:
results = []

# Each training set is sampled independently; only the test set is shared.
DIAG_TRAIN_SIZES = [5000, 26000, 58000]

for data_seed in range(1):
    for train_size in DIAG_TRAIN_SIZES:
        for noise_level in EFFECTIVE_NOISE_GRID:
            for cov in COV_GRID:
                true_edges_s, splits = prepare_data_fixed_test(
                    generate,
                    train_size=train_size,
                    data_seed=data_seed,
                    test_size=TEST_SIZE,
                    valid_size=VALID_SIZE,
                    test_seed=TEST_SEED + data_seed,
                    cov=cov,
                    noise_level=noise_level,
                    feature_selection=FEATURE_SELECTION,
                )

                X_train_s, y_train_s = splits[0], splits[3]
                ii_s = compute_info(X_train_s, y_train_s, n_bins=N_BINS)
                graph_true_s = build_true_graph(ii_s, true_edges_s)
                n_edges = len(graph_true_s.edges()) or 10

                ii_graph = build_ii_graphs(
                    ii_s,
                    PERMUTE_SEEDS,
                    n_edges=n_edges,
                    n_bins=N_BINS,
                )

                # Compare each estimated graph with the known interaction graph.
                for graph_name, stats in graph_stats(ii_graph, true_edges_s).items():
                    results.append(
                        {
                            "cov": cov,
                            "noise_level": noise_level,
                            "data_seed": data_seed,
                            "n_samples": train_size,
                            "graph_name": graph_name,
                            "precision": stats["precision"],
                            "recall": stats["recall"],
                            "edges_len": stats["edges_len"],
                        }
                    )

results_df = pd.DataFrame(results)
results_df

interaction_ranking = (
    ii_s.melt(ignore_index=False)
    .reset_index()
    .rename(
        columns={
            "index": "feature_1",
            "variable": "feature_2",
            "value": "interaction_information",
        }
    )
)
interaction_ranking = interaction_ranking[
    interaction_ranking["feature_1"] < interaction_ranking["feature_2"]
].copy()
interaction_ranking = (
    interaction_ranking.sort_values("interaction_information", ascending=False)
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "rank"})
)
interaction_ranking["rank_fraction"] = (
    interaction_ranking["rank"].cumsum() / interaction_ranking["rank"].sum()
)
interaction_ranking["pair"] = list(
    zip(interaction_ranking["feature_1"], interaction_ranking["feature_2"])
)
interaction_ranking["true_interaction"] = interaction_ranking["pair"].isin(true_edges_s)
interaction_ranking[interaction_ranking["true_interaction"]].head(20)

### Diagnostic plots

In [ ]:
y_var = "interaction_information"
x_var = "rank"

plot_df = interaction_ranking.sort_values(y_var, ascending=False).reset_index(drop=True)
plot_df["rank_pct"] = plot_df["rank"] / len(plot_df)

px.scatter(
    plot_df,
    x=x_var,
    y=y_var,
    color="true_interaction",
)

In [ ]:
df_plot = results_df[results_df["graph_name"] == "ii"].copy()

for metric in ["precision", "recall"]:
    heatmap_data = df_plot.pivot(
        index="n_samples",
        columns="cov",
        values=metric,
    )
    sns.heatmap(
        heatmap_data,
        cmap="coolwarm",
        center=0,
        annot=True,
        fmt=".2f",
    )
    plt.title(metric)
    plt.show()

In [ ]:
for variable in ["cov", "noise_level"]:
    for metric in ["precision", "recall"]:
        sns.lineplot(
            data=results_df,
            x=variable,
            y=metric,
            hue="graph_name",
            style="graph_name",
            markers=True,
            dashes=False,
        )
        plt.show()

## Hyperparameter sweep and evaluation

Review the resolved configuration, then tune MLP, GNN, and XGBoost models. The sweep
logs trials to Optuna and MLflow and evaluates the selected models on the held-out test
set.

In [ ]:
print(f"Generator: {GENERATOR_NAME}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Train sizes: {TRAIN_SIZE_GRID}")
print(f"Data seeds: {DATA_SEEDS}")
print(f"Covariance grid: {COV_GRID}")
print(f"Noise grid: {NOISE_GRID} (effective: {EFFECTIVE_NOISE_GRID})")

In [ ]:
results = run_sweep(
    generate=generate,
    permute_seeds=PERMUTE_SEEDS,
    noise_grid=EFFECTIVE_NOISE_GRID,
    cov_grid=COV_GRID,
    train_size_grid=TRAIN_SIZE_GRID,
    data_seeds=DATA_SEEDS,
    main_data_seed=DATA_SEED_MAIN,
    edge_mode=EDGE_MODE,
    threshold_grid=THRESHOLD_GRID,
    n_edges_grid=N_EDGES_GRID,
    model_configs=MODEL_CONFIGS,
    technical_settings=technical_settings,
    direction=DIRECTION,
    n_trials=N_TRIALS,
    n_bins=N_BINS,
    test_size=TEST_SIZE,
    valid_size=VALID_SIZE,
    top_k=TOP_K,
    experiment_name=EXPERIMENT_NAME,
    generator_name=GENERATOR_NAME,
    config_hash=GENERATOR_HASH,
    generator_kwargs=GENERATOR_KWARGS,
    optuna_storage=OPTUNA_STORAGE,
    standardize=STANDARDIZE,
    keep_on_gpu=KEEP_ON_GPU,
    n_startup_trials=N_STARTUP_TRIALS,
    interactions_names=(
        INTERACTIONS_NAMES if "interactions" in GENERATOR_KWARGS else None
    ),
    xgb_search_space=XGB_SEARCH_SPACE,
    test_seed=TEST_SEED,
    permuted_top_k=None,
    preprocessing_pipeline=PIPELINE,
    graph_preprocessing_pipeline=PIPELINE_DISCRETIZED,
)

In [ ]:
results_df = pd.DataFrame(results)
results_df